SCALCS is designed to calculate and display variuos properties of ion channels as described in several papers (see references in SCALCS main page).

##### Some general settings

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
my_colour = ["r", "g", "b", "m", "c", "y"]
import numpy as np

In [2]:
from scalcs import scalcsio
from scalcs.samples import samples
from scalcs import version
from scalcs import scalcslib as scl
from scalcs import scplotlib as scpl
from scalcs import popen
from scalcs import scburst
from scalcs import cjumps

ModuleNotFoundError: No module named 'scalcs'

##### Define a mechanism

First, all calculations here require to specify a mechanism and give values for the rate constants (i.e. to define the $Q$ matrix).  Different ways to do that are shown elsehere (... when ready ...).  
In this notebook a mechanism used as the numerical example by Colquhoun & Hawkes 1982 is loaded from DCPYPS. 

In [ ]:
demomec = samples.CH82()
print(demomec)

Define temporal resolution, $t_{res}$, for current calculations.

In [ ]:
tres = 50e-6  # resolution in seconds

Set agonist concentration:

In [ ]:
conc = 100e-9    # 100 nM
demomec.set_eff('c', conc)

Calculate and display equilibrium occupancies and lifetime of states.

In [ ]:
print(scl.printout_occupancies(demomec, tres))

##### Open and shut time distributions

Calculate ideal and asymptotic distributions for both open and shut dwell times. 

In [ ]:
print(scl.printout_distributions(demomec, tres))

Display open time distribution:

In [ ]:
t, ipdf, epdf, apdf = scpl.open_time_pdf(demomec, tres)
plt.semilogx(t, ipdf, 'r--', t, epdf, 'b-', t, apdf, 'g-')
plt.ylabel('fopen(t)')
plt.xlabel('Open time, ms')
plt.title('The open time pdf')
print('RED- ideal distribution\nGREEN- HJC distribution (corrected for missed events)')

Display shut time distribution:

In [ ]:
t, ipdf, epdf, apdf = scpl.shut_time_pdf(demomec, tres)
plt.semilogx(t, ipdf, 'r--', t, epdf, 'b-', t, apdf, 'g-')
plt.ylabel('fshut(t)')
plt.xlabel('Shut time, ms')
plt.title('The shut time pdf')
print('RED- ideal distribution\nGREEN- HJC distribution (corrected for missed events)')

Calculate and display open, shut and open/shut time correlations:

In [ ]:
print(scl.printout_correlations(demomec))

In [ ]:
lag = 5
n, roA, roF, roAF = scpl.corr_open_shut(demomec, lag)
plt.plot(n, roA,'go', n, roF, 'ro', n, roAF, 'bo')
plt.axhline(y=0, xmin=0, xmax=1, color='k')
plt.xlim([0, 6])
print( 'Shut time correlation - red circles.\n' +
    'Open time correlation - green circles\n' +
    'Open-shut time correlation - blue circles')

Display open time adjacent to shut time range pdf:

In [ ]:
u1, u2 = 0.1e-3, 1e-3 # 1 ms, 10 ms
t, ipdf, ajpdf = scpl.adjacent_open_time_pdf(demomec, tres, u1, u2)
plt.semilogx(t, ipdf, 'r--', t, ajpdf, 'b-')
print(scl.printout_adjacent(demomec, u1, u2))
print('Ideal open time pdf- red dashed line.\n' +
'Open times adjacent to shut time range pdf- blue solid line.\n')

Display mean open time preceding / next-to shut time plot:

In [ ]:
sht, mp, mn = scpl.mean_open_next_shut(demomec, tres)
plt.semilogx(sht, mp, 'r--', sht, mn, 'b--')
print('Mean open time preceding specified shut time- red dashed line.\n' +
'Mean open time next to specified shut time- blue dashed line.')

Display subset time pdf:

In [ ]:
state1 = 3
state2 = 4
#t, ipdf, spdf = scpl.subset_time_pdf(demomec, tres, state1, state2)
#plt.semilogx(t, spdf, 'b-', t, ipdf, 'r--')
print('Ideal pdf- red dashed line.\nSubset life time pdf- blue solid line.')

Display dependency plot:

In [ ]:
to, ts, d = scpl.dependency_plot(demomec, tres, points=128)
fig = plt.figure()
fig.suptitle('Dependency plot', fontsize=12)
ax = fig.add_subplot(111, projection='3d')
to, ts = np.meshgrid(to, ts)
surf = ax.plot_surface(to, ts, d, rstride=1, cstride=1, cmap=cm.coolwarm,
    linewidth=0, antialiased=False)
ax.set_zlim(-1.0, 1.0)

##### Burst properties

Calculate burst properties

In [ ]:
print('Agonist concentration = %e M' %conc)
print(scburst.printout_pdfs(demomec))

Display burst length distribution:

In [ ]:
t, fbst = scpl.burst_length_pdf(demomec)
plt.semilogx(t, fbst, 'b-')
plt.ylabel('fbst(t)')
plt.xlabel('burst length, ms')
plt.title('The burst length pdf')

Display the conditional burst length distribution:

In [ ]:
t, fbst, cfbst = scpl.burst_length_pdf(demomec, conditional=True)
plots = []
for i in range(demomec.kA):
    handle, = plt.semilogx(t, cfbst[i], my_colour[i]+'-', label="State {0:d}".format(i+1))
    plots.append(handle)
handle, = plt.semilogx(t, fbst, 'k-', label="Not conditional")
plots.append(handle)
plt.legend(handles=plots)

Display the distribution of number of openings per burst:

In [ ]:
n = 10
r, Pr = scpl.burst_openings_pdf(demomec, n)
plt.plot(r, Pr,'ro')
plt.xlim([0, 11])

 Display the conditional distribution of number of openings per burst:

In [ ]:
n = 10
r, Pr, cPr = scpl.burst_openings_pdf(demomec, n, conditional=True)
plots = []
for i in range(demomec.kA):
    handle, = plt.plot(r, cPr[i], my_colour[i]+'o', label="State {0:d}".format(i+1))
    plots.append(handle)
handle, = plt.plot(r, Pr,'ko', label="Not conditional")
plots.append(handle)
plt.legend(handles=plots)
plt.xlim([0, n+1])

  Display mean burst length versus concentration plot:

In [ ]:
cmin = 10e-9
cmax = 1e-3
c, br, brblk = scpl.burst_length_versus_conc_plot(demomec, cmin, cmax)
plt.plot(c, br,'r-')
print('Solid line: mean burst length versus concentration.' + '    X-axis: microMols; Y-axis: ms.')

##### $P_{open}$ curve

Calculate Popen curve parameters:

In [ ]:
print(popen.printout(demomec, tres))

Display $P_{open}$ curve:

In [ ]:
c, pe, pi = scpl.Popen(demomec, tres)
plt.semilogx(c, pe, 'b-', c, pi, 'r--')
plt.ylabel('Popen')
plt.xlabel('Concentration, M')
plt.title('Apparent and ideal Popen curves')
print('RED- ideal curve\nBLUE- apparent curve (corrected for missed events)')

##### Macroscopic response to agonist concentration pulse

Define a realistic profile of concentration pulse:

In [ ]:
# Here one can tweak the parameters of the jump.
step_size = 8e-6 # The sample step. All time parameters in seconds
pulse_centre = 10e-3
rise_time = 250e-6 # 10-90% rise time for error functions
pulse_width = 10e-3
record_length = 50e-3
peak_conc = 10e-6    # in M
baseline_conc = 0.0
cjargs = (peak_conc, baseline_conc, pulse_centre, pulse_width,
            rise_time, rise_time)

In [ ]:
t, c, Popen, P  = cjumps.solve_jump(demomec, record_length, step_size,
        cjumps.pulse_erf, cjargs)
maxP = max(Popen)
maxC = max(c)
c1 = (c / maxC) * 0.2 * maxP + 1.02 * maxP

plt.plot(t * 1000, Popen,'b-', t * 1000, c1, 'g-')
plt.ylabel('Open probability')
plt.xlabel('Time, ms')
plt.title('Concentration jump')
print('GREEN- concentration pulse profile\nBLUE- open probability profile')

Calculate properties of a macroscopic response to an ideal square pulse:

In [ ]:
print (cjumps.printout(demomec, peak_conc, pulse_width))